In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\Raw_data_1Day_2024_site_114_IHBAS_Dilshad_Garden_Delhi_CPCB_1Day.csv")

In [3]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,169.57,305.14,28.77,29.65,38.54,62.22,52.05,0.77,33.90,...,2.06,NaN,78.67,1.22,133.79,NaN,0.0,33.45,NaN,-0.22
1,2024-01-02,148.21,288.49,29.95,27.63,38.40,59.98,29.71,0.78,32.73,...,2.06,NaN,73.04,1.13,131.23,NaN,0.0,42.14,NaN,-0.22
2,2024-01-03,142.83,266.59,31.23,29.64,40.49,56.22,25.18,1.14,32.70,...,2.04,NaN,91.74,1.05,152.68,NaN,0.0,24.69,NaN,-0.25
3,2024-01-04,183.10,343.50,37.54,33.01,47.30,56.74,21.95,0.88,36.09,...,2.05,NaN,88.92,1.13,86.84,NaN,0.0,21.57,NaN,-0.26
4,2024-01-05,143.22,254.06,36.58,33.53,46.82,40.75,31.97,0.55,28.09,...,3.29,NaN,94.04,1.02,109.93,NaN,0.0,12.37,NaN,-0.26
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,52.04,170.73,17.07,33.67,31.78,48.21,9.23,1.23,49.00,...,1.77,NaN,90.55,0.90,161.97,NaN,0.0,17.69,NaN,-0.06
362,2024-12-28,22.37,84.67,15.54,34.20,30.81,47.70,9.08,1.57,49.20,...,1.77,NaN,94.48,0.78,119.71,NaN,0.0,25.07,NaN,-0.02
363,2024-12-29,33.05,113.79,16.05,34.62,31.46,48.10,8.54,0.90,48.67,...,1.80,NaN,89.75,1.28,87.89,NaN,0.0,58.57,NaN,-0.03
364,2024-12-30,28.77,96.14,17.07,33.23,31.54,48.14,8.48,1.01,48.87,...,1.80,NaN,82.92,1.12,79.84,NaN,0.0,46.83,NaN,-0.08


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (366, 22)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['O Xylene (µg/m³)']
Dropped rows (>70% NaN): 0
Missing values after imputation:
 Timestamp              0
PM2.5 (µg/m³)          0
PM10 (µg/m³)           0
NO (µg/m³)             0
NO2 (µg/m³)            0
NOx (ppb)              0
NH3 (µg/m³)            0
SO2 (µg/m³)            0
CO (mg/m³)             0
Ozone (µg/m³)          0
Benzene (µg/m³)        0
Toluene (µg/m³)        0
Xylene (µg/m³)         0
Eth-Benzene (µg/m³)    0
MP-Xylene (µg/m³)      0
RH (%)                 0
WS (m/s)               0
WD (deg)               0
TOT-RF (mm)            0
SR (W/mt2)             0
VWS (m/s)              0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (366, 21)
    Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-01-01         169.57        305.14       28.77        29.65   
1  2024-01-02         148.21        288.49       29.95        27.63   
2  2024-01-03         142.83        266.59       31.23        29.64   
3  2024-01-04          65.90        343.50       37.54        33.01   
4  2024-01-05         143.22        254.06       36.58        33.53   

   NOx (ppb)  NH3 (µg/m³)  SO2 (µg/m³)  CO (mg/m³)  Ozone (µg/m³)  ...  \
0      38.54        62.22        14.41        0.77          33.90  ...   
1      38.40        59.98        29.71        0.78          32.73  ...   
2      40.49        56.22        25.18        1.14          32.70  ...   
3      47.30        56.74        21.95        0.88          36.09  ...   
4      46.82        40.75        31.97        0.55          28.09  ...   

   Toluene (µg/m³)  Xylene (µg/m³)  Eth-Benzene (µg/m³)  MP-Xylene (µg/m³)  \
0             4.55         

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,Toluene (µg/m³),Xylene (µg/m³),Eth-Benzene (µg/m³),MP-Xylene (µg/m³),RH (%),WS (m/s),WD (deg),TOT-RF (mm),SR (W/mt2),VWS (m/s)
0,2024-01-01,2.835638,1.818965,0.024163,-0.065690,-0.121661,2.673387,-0.039063,-0.897352,1.923993,...,-0.297417,-0.045722,0.163860,-0.214620,0.954771,0.352579,-0.365014,0.0,-1.825337,-0.960055
1,2024-01-02,2.240156,1.607065,0.147468,-0.253018,-0.143911,2.492628,2.308246,-0.876198,1.753262,...,-0.293858,-0.040464,0.163860,-0.214620,0.706544,0.042128,-0.411182,0.0,-1.533682,-0.960055
2,2024-01-03,2.090171,1.328349,0.281223,-0.066617,0.188256,2.189212,1.613258,-0.114669,1.748885,...,-0.304535,-0.045722,0.137058,-0.229645,1.531028,-0.233827,-0.024345,0.0,-2.119342,-1.190116
3,2024-01-04,-0.054510,2.307164,0.940593,0.245906,1.270581,2.231174,1.117715,-0.664662,2.243565,...,-0.297417,-0.040464,0.169220,-0.222132,1.406694,0.042128,-1.211728,0.0,-2.224056,-1.266803
4,2024-01-05,2.101043,1.168882,0.840277,0.294129,1.194294,0.940846,2.654972,-1.362731,1.076178,...,0.197255,0.574744,1.032254,0.709420,1.632435,-0.337311,-0.795314,0.0,-2.532828,-1.266803
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,-0.440904,0.108362,-1.198442,0.307112,-1.196040,1.542837,-0.833773,0.075714,-0.037218,...,-0.891736,-0.208725,-0.688452,-0.432483,1.478561,-0.751244,0.143194,0.0,-2.354277,0.266938
362,2024-12-28,-1.268054,-0.986903,-1.358321,0.356263,-1.350203,1.501683,-0.856786,0.794936,-0.037218,...,-0.891736,-0.208725,-0.699173,-0.432483,1.651835,-1.165178,-0.618938,0.0,-2.106588,0.573686
363,2024-12-29,-0.970313,-0.616300,-1.305028,0.395212,-1.246898,1.533961,-0.939632,-0.622355,-0.037218,...,-0.884618,-0.213984,-0.699173,-0.409945,1.443289,0.559545,-1.192792,0.0,-0.982256,0.496999
364,2024-12-30,-1.089633,-0.840927,-1.198442,0.266308,-1.234183,1.537189,-0.948837,-0.389665,-0.037218,...,-0.891736,-0.208725,-0.704534,-0.409945,1.142154,0.007634,-1.337968,0.0,-1.376276,0.113564


In [10]:
df.to_excel('IHBAS2024.xlsx', index=False)